# `GenVarLoader`

In [1]:
# Automatically reload code in notebook
%load_ext autoreload
%autoreload 2

import os
os.chdir(os.path.dirname(os.path.abspath('.')))
import pandas as pd
import polars as pl
import seqpro as sp
import genvarloader as gvl


import src.utils as utils
import src.config as config
import src.genvarloader as GVL
import src.onekg as og
# import src.pyensembl as PYE

/grid/koo/home/schilder/.conda/envs/genome-loader/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
example_paths = GVL.prepare_example()

In [2]:
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import genvarloader as gvl
import numba as nb
import numpy as np
import polars as pl
import seqpro as sp
import pooch
from loguru import logger
from einops import rearrange
from tqdm.auto import tqdm

In [3]:
# GRCh38 chromosome 22 sequence
reference = pooch.retrieve(
    url="https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.22.fa.gz",
    known_hash="sha256:974f97ac8ef7ffae971b63b47608feda327403be40c27e391ee4a1a78b800df5",
    progressbar=True,
)
if not Path(f"{reference[:-3]}.bgz").exists():
    !gzip -dc {reference} | bgzip > {reference[:-3]}.bgz
reference = reference[:-3] + ".bgz"

# PLINK 2 files
variants = pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pgen",
    known_hash="md5:31aba970e35f816701b2b99118dfc2aa",
    progressbar=True,
    fname="1kGP.chr22.pgen",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.psam",
    known_hash="md5:eefa7aad5acffe62bf41df0a4600129c",
    progressbar=True,
    fname="1kGP.chr22.psam",
)
pooch.retrieve(
    url="doi:10.5281/zenodo.13656224/1kGP.chr22.pvar",
    known_hash="md5:5f922af91c1a2f6822e2f1bb4469d12b",
    progressbar=True,
    fname="1kGP.chr22.pvar",
)


In [12]:
bed = gvl.read_bedlike("notebooks/snps_500kb_windows.bed")[:20]
bed.head()

chrom,chromStart,chromEnd,name,score,strand
str,i64,i64,str,f64,str
"""22""",16835042,17335042,"""340557""",0.0,"""+"""
"""22""",16835050,17335050,"""340558""",0.0,"""+"""
"""22""",16835053,17335053,"""340559""",0.0,"""+"""
"""22""",16835059,17335059,"""894942""",0.0,"""+"""
"""22""",16835065,17335065,"""340560""",0.0,"""+"""


## Splicing validation

### import Haplotypes: Haplosaurus







### 1. Get transcript coordinates

Get transcript/exon coordinates from Ensembl (release 113) GTF:
https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/

In [4]:
fn="https://ftp.ensembl.org/pub/release-113/gtf/homo_sapiens/Homo_sapiens.GRCh38.113.chr_patch_hapl_scaff.gtf.gz"

In [ ]:
import gffutils

dbfn = os.path.join(os.path.expanduser('~/.cache')  , 
                    os.path.basename(fn).replace('.gtf.gz', '.db'))
db = gffutils.create_db(fn, 
                        dbfn=dbfn, 
                        force=True, 
                        keep_order=True, 
                        merge_strategy='merge', 
                        sort_attribute_values=True, 
                        disable_infer_genes=True, 
                        disable_infer_transcripts=True)
annotations_db = gffutils.FeatureDB(db)


## `cyvcf2`

In [ ]:
import numpy as np
from cyvcf2 import VCF

def vcf_to_genotype_matrix(vcf_file, region):
    """
    Converts a VCF file to a genotype matrix for a specific region.

    Args:
        vcf_file (str): Path to the VCF file.
        region (str): Region to extract (e.g., "chr1:1000-2000").

    Returns:
        tuple: A tuple containing:
            - genotype_matrix (numpy.ndarray): Genotype matrix (samples x variants).
            - variant_ids (list): List of variant IDs.
            - sample_ids (list): List of sample IDs.
    """
    vcf = VCF(vcf_file)
    sample_ids = vcf.samples
    genotype_matrix = []
    variant_ids = []

    for variant in vcf(region):
        variant_ids.append(variant.ID)
        genotypes = [
            sum(gt) if gt != [-1, -1] else np.nan for gt in variant.genotypes
        ]
        genotype_matrix.append(genotypes)

    vcf.close()
    return np.array(genotype_matrix).T, variant_ids, sample_ids

vcf_file_path = "/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
genotype_matrix, sample_names, variant_ids = vcf_to_genotype_matrix(vcf_file_path, "22:10664523-10684523")
print("Genotype Matrix:")
print(genotype_matrix)
print("\nSample Names:", sample_names)
print("\nVariant IDs:", variant_ids)

## `sgkit`

In [4]:
import sgkit as sg

In [5]:
ds = sg.simulate_genotype_call_dataset(n_variant=100, n_sample=50, missing_pct=.1)

In [ ]:
ds[['variant_allele', 'call_genotype']]


In [ ]:
ds.keys()

In [ ]:
df = ds.to_dataframe().reset_index()
df.head()

In [ ]:
# Create a new column by combining position and allele information
df['variant_id'] = df['variant_position'].astype(str) + df['variant_allele'].astype(str) + df['call_genotype'].astype(str)

# Create a ploidy-specific sample_id column
df['sample_id_ploidy'] = df['sample_id'].astype(str) + '_' + df['ploidy'].astype(str)

df.head()

Create a string representing the set of variants present in a given sample (one per ploid).

Then encode that string as an md5 hash for compression.

In [ ]:
import hashlib
df_agg = df.groupby('sample_id_ploidy')['variant_id'].apply(lambda x: ','.join(x)).reset_index()
df_agg['haplotype_hash'] = df_agg['variant_id'].apply(lambda x: hashlib.md5(x.encode()).hexdigest())
df_agg.head()


Create a hashmap for each haplotype hash to a list of sample_ids.

In [ ]:
# Group by haplotype_hash and get a list of sample_ids for each hash
haplotype_samples = df_agg.groupby('haplotype_hash')['sample_id_ploidy'].apply(list).reset_index()
print(haplotype_samples.shape)
haplotype_samples

In [ ]:
sg.display_genotypes(ds, max_variants=10, max_samples=10)

In [ ]:
sg.convert_call_to_index(ds).call_genotype_index.values


## `haptools`

`haptools` provides some convenient data structures and functions for working with haplotypes. BUT it does not actually create haplotype file for you.  
Users must generate these themselves beforehand and figure out how to convert it into the haptools-specific `.hap` file format.

PLINK can generate haplotype blocks, but this is a data-driven approach (using linkage disequilibrium-clumping information from a popuatlion) that does not produce one block of a given window size (as models like SpliceAI expect).
https://www.cog-genomics.org/plink/1.9/ld#blocks

In [12]:
from haptools import data

In [13]:
vcf_file_path="/grid/koo/home/schilder/projects/data/1KG/vcf/ALL.chr22.shapeit2_integrated_snvindels_v2a_27022019.GRCh38.phased.vcf.gz"
gt = data.GenotypesVCF(vcf_file_path) 

In [ ]:
print(gt.variants[:10])
print(gt.samples[:10])

In [ ]:
genotypes = gt.read(region="22:10664523-10764523")

In [ ]:
for i,line in enumerate(gt.__iter__(region="22:10664523-10764523")):
    print(line)
    if i > 5:
        break

In [ ]:
!haptools transform --region "22:10664523-10764523" {vcf_file_path} {vcf_file_path.replace('.vcf.gz', '.hap')}
